# 📊 RAGAS Evaluation: Measuring Hybrid RAG Quality

This notebook demonstrates how to evaluate the performance of our **Hybrid RAG Retriever** (combining **ChromaDB** dense embeddings and **BM25** sparse keyword matching) using the **RAGAS** (Retrieval Augmented Generation Assessment) framework.

Evaluating retrieval and generation quality is a critical step in building production-ready LLM systems.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# Load environment keys
load_dotenv(dotenv_path=".env")
api_key = os.getenv("API_KEY") or os.getenv("GEMINI_API_KEY")

## 1. Setup Sample Corpus & Hybrid Retriever

We will index a small text corpus to test keyword (BM25) and semantic (Chroma) retrieval paths.

In [ ]:
sample_texts = [
    "ChatSphere is a modular chatbot built on LangGraph with DuckDuckGo search, a secure calculator, and Chroma vector database.",
    "ChromaDB is a local, persistent vector database used to store document embeddings for RAG systems.",
    "LangGraph structures agentic execution loops as state graphs with nodes and conditional edges.",
    "LangSmith provides tracing, observability, and debugging dashboards to track token usage and latencies."
]

docs = [Document(page_content=text, metadata={"source": "sample_docs"}) for text in sample_texts]

# Setup Embeddings and Chroma DB vector store
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)
vector_store = Chroma.from_documents(docs, embeddings)

# Dense retriever
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Sparse BM25 retriever
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 2

# Ensemble Hybrid retriever (RRF weighting)
hybrid_retriever = EnsembleRetriever(
    retrievers=[sparse_retriever, dense_retriever],
    weights=[0.4, 0.6]
)

## 2. Generate Evaluation Testset

We will evaluate queries by retrieving the contexts from the retriever, and generating model answers.

In [ ]:
eval_questions = [
    "What is ChatSphere built on?",
    "What database does ChatSphere use?",
    "How does LangGraph structure execution loops?"
]

ground_truths = [
    "ChatSphere is built on LangGraph.",
    "ChatSphere uses Chroma vector database.",
    "LangGraph structures execution loops as state graphs with nodes and conditional edges."
]

eval_data = []
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key, temperature=0)

for question, gt in zip(eval_questions, ground_truths):
    # Retrieve context
    retrieved_docs = hybrid_retriever.invoke(question)
    contexts = [doc.page_content for doc in retrieved_docs]
    
    # Generate response
    prompt = f"Contexts:\n{chr(10).join(contexts)}\n\nQuestion: {question}\nAnswer:"
    response = llm.invoke(prompt)
    
    eval_data.append({
        "question": question,
        "contexts": contexts,
        "answer": response.content,
        "ground_truth": gt
    })

eval_df = pd.DataFrame(eval_data)
eval_df

## 3. Run RAGAS Assessment

Ragas evaluates RAG systems across four core dimensions:
- **Faithfulness**: Is the answer derived *only* from the context?
- **Answer Relevancy**: Does the answer directly address the question?
- **Context Recall**: Are the retrieved contexts sufficient to reconstruct the ground truth?

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall

# Convert to HuggingFace dataset format expected by Ragas
dataset = Dataset.from_pandas(eval_df)

# Setup ragas evaluator LLM
ragas_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)

result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_recall],
    llm=ragas_llm
)

print("\n=== Ragas Evaluation Report ===")
print(result)